In [1]:
import re

def clean_text(text: str) -> str:
    """Remove newlines, tabs, extra spaces, punctuation and lowercase."""
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[!?]+", "", text)
    return text.strip().lower()

In [2]:
import random
import sqlite3
import re

conn = sqlite3.connect(r"E:\Github\LawAssistant\triplet_extraction\batch_preprocessing\GTVT_law.db")
cursor = conn.cursor()

def extract_all_with_parents(cursor):
    # Step 1: Find all "Điểm" nodes
    cursor.execute("SELECT * FROM laws WHERE title LIKE '%Điểm%'")
    rows = cursor.fetchall()
    if not rows:
        return []

    columns = [col[0] for col in cursor.description]
    results = []

    # Step 2: For each node, find its parent chain
    for row in rows:
        node = dict(zip(columns, row))
        parent_chain = []

        current = node
        visited = set()

        while current['parent_id'] is not None and current['parent_id'] not in visited:
            visited.add(current['id'])
            cursor.execute("SELECT * FROM laws WHERE id = ?", (current['parent_id'],))
            parent = cursor.fetchone()
            if not parent:
                break
            parent_dict = dict(zip(columns, parent))
            parent_chain.append(parent_dict)
            current = parent_dict

        results.append({
            "target": node,
            "ancestors": parent_chain[::-1]
        })

    return results

In [4]:
diem_db_data = extract_all_with_parents(cursor)
diem_data = {}

for diem in diem_db_data:
    content = ""
    title = ""

    for r in diem["ancestors"]:
        title += r['title'] + " "
        if not (r['title'].strip().startswith("Chương") or r['title'].strip().startswith("Điều")):
            content += r['content'] + "\n"

    title += diem['target']["title"]
    content += diem['target']["content"]

    so_hieu = diem['target']['so_hieu']
    ID = diem['target']['id']

    diem_data[ID] = {
        "so_hieu": so_hieu,
        "title": title,
        "content": content
    }

In [5]:
from openai import OpenAI
from dotenv import load_dotenv
import os

# Load .env file
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

def get_gpt_response(user_prompt, system_prompt, api_key, model="gpt-4o"):
    client = OpenAI(api_key=api_key)
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
    )
    print(response.choices[0].message.content)
    return response.choices[0].message.content

In [6]:
def create_prompt(so_hieu, content, clean_text):
    system_prompt_rewrite = """
Bạn là trợ lý AI Tiếng Việt chuyên nghiệp và trung thực.
Bạn là chuyên gia pháp luật Việt Nam, am hiểu các bộ luật, nghị định, và văn bản pháp luật.
Bạn là chuyên gia ngôn ngữ Việt Nam, biết viết câu chuẩn cấu trúc, chính xác, trang trọng, và đúng ngôn ngữ pháp lý.
Luôn trả lời chính xác, hữu ích, ngắn gọn và an toàn.
Không thay đổi ý nghĩa khi viết lại câu.
Luôn dùng ngôn ngữ chính xác như trong văn bản pháp luật, tránh ngôn ngữ thông thường hay không trang trọng.
Định nghĩa 'câu đơn': Một câu đơn là câu có một chủ ngữ (hoặc cụm chủ ngữ) và một vị ngữ (hoặc cụm vị ngữ), biểu đạt một ý trọn vẹn; câu có thể chứa thành tố phụ (tính từ, trạng từ, bổ ngữ) nhưng không được ghép bằng liên từ hoặc dấu câu như dấu ",", ";" để tạo hai hoặc nhiều mệnh đề độc lập.

Quy tắc bắt buộc:
1. Khi viết lại, chỉ trả về các câu đơn theo đúng định nghĩa trên; mỗi câu một dòng nếu có nhiều câu.
2. Được phép tái sử dụng các thành phần câu (chủ ngữ, cụm danh từ, đại từ, cụm tính từ, v.v.) từ vế trước hoặc từ phần khác của câu gốc để hoàn chỉnh vế thiếu, nhằm bảo toàn ý nghĩa sau khi tách.
3. Khi tái sử dụng, ưu tiên giữ nguyên** từ ngữ gốc; chỉ thực hiện điều chỉnh nhỏ cần thiết để tạo câu đơn ngữ pháp đúng, **không** thêm thông tin, suy đoán hay nội dung mới.
4. Tuyệt đối không kèm chú giải, giải thích, danh sách hay bất kỳ nội dung nào khác ngoài các câu viết lại.
5. Nếu câu gốc mơ hồ hoặc thiếu thông tin đến mức không thể tạo câu đơn hoàn chỉnh mà vẫn giữ nguyên ý, hãy yêu cầu thêm thông tin ngắn gọn.
Danh mục liên từ cần loại trừ khi viết câu đơn: và, hoặc, hoặc là, hay, hay là, nhưng, song, tuy nhiên, mà, còn, rồi.
Giữ nguyên thứ tự trước sau của các từ sau khi viết lại câu.
Sau khi viết lại câu không được thiếu từ danh từ nào trong câu gốc và phải độc lập không phụ thuộc vào câu trước đó.
"""


    user_prompt_rewrite = f"""
Ngữ cảnh: Bộ luật số {so_hieu} trong luật Việt Nam
Nhiệm vụ: Viết lại câu sau để hoàn chỉnh cấu trúc với đầy đủ chủ ngữ và vị ngữ, giữ nguyên ý nghĩa. Mỗi câu xuất ra phải là một câu đơn đầy đủ (một dòng một câu nếu có nhiều câu).
Câu cần viết lại: "{clean_text(content)}"
"""
    return system_prompt_rewrite, user_prompt_rewrite

In [16]:
import json

list_of_prompts = {}

for key, value in diem_data.items():
    system_prompt, user_prompt = create_prompt(
        value['so_hieu'],
        value['content'],
        clean_text
    )
    list_of_prompts[key] = (system_prompt, user_prompt)

tasks = []
for key, prompt in list_of_prompts.items():
    task = {
        "custom_id": key,
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": "gpt-4.1-mini",
            "messages": [
                { "role": "system", "content": prompt[0]},
                { "role": "user", "content": prompt[1]}
            ]
        }
    }
    tasks.append(task)

with open("batch_tasks.jsonl", "w") as f:
    for t in tasks:
        f.write(json.dumps(t) + "\n")

In [17]:
client = OpenAI(api_key=api_key)
batch_file = client.files.create(
    file=open("batch_tasks.jsonl", "rb"),
    purpose="batch"
)

batch_job = client.batches.create(
    input_file_id=batch_file.id,
    endpoint="/v1/chat/completions",
    completion_window="24h"
)

In [3]:
import json

with open("batch_68f0720bbe3081908aa61019a6d518fe_output.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        if data.get("response", {}).get("status_code") != 200:
            continue
        content = data["response"]["body"]["choices"][0]["message"]["content"]
        print("────────────────────────────")
        print("🆔", data["custom_id"])
        print(content)

Việc quản lý các hạng mục công trình dưới đây được thực hiện theo quy định của thông tư này và quy định của pháp luật có liên quan.  
Việc khai thác các hạng mục công trình dưới đây được thực hiện theo quy định của thông tư này và quy định của pháp luật có liên quan.  
Việc bảo trì các hạng mục công trình dưới đây được thực hiện theo quy định của thông tư này và quy định của pháp luật có liên quan.  
Việc quản lý công trình dân dụng được thực hiện theo quy định của pháp luật về công trình dân dụng.  
Việc quản lý công trình hạ tầng kỹ thuật đô thị được thực hiện theo quy định của pháp luật về công trình dân dụng.  
Việc quản lý công trình công nghiệp vật liệu xây dựng được thực hiện theo quy định của pháp luật.  
Việc quản lý công trình hạ tầng kỹ thuật đô thị được thực hiện theo quy định của pháp luật.


## Lưu kết quả xử lý vào DB

In [2]:
import json
from tqdm import tqdm
import sqlite3

conn = sqlite3.connect("process_law.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS laws (
    id TEXT,
    sequence INTEGER,
    content TEXT,
    so_hieu TEXT NOT NULL,
    PRIMARY KEY (id, sequence)
);
""")
conn.commit()


In [3]:
# Paths
db_path = r"E:\Github\LawAssistant\triplet_extraction\batch_preprocessing\process_law.db"
batch_result_path = r"E:\Github\LawAssistant\triplet_extraction\batch_68f0720bbe3081908aa61019a6d518fe_output.jsonl"

# Initialize database connection
conn = sqlite3.connect(db_path)

cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print("Các bảng trong law.db:", tables)

Các bảng trong law.db: [('laws',)]


In [4]:
import sqlite3
gtvt_conn = sqlite3.connect(r"E:\Github\LawAssistant\triplet_extraction\batch_preprocessing\GTVT_law.db")
gtvt_cursor = gtvt_conn.cursor()

In [5]:
def extract_from_sqlite(cursor, id: int, include_parent=False):
    cursor.execute("SELECT * FROM laws WHERE id = ?", (id,))
    rows = cursor.fetchall()

    columns = [col[0] for col in cursor.description]
    results = [dict(zip(columns, row)) for row in rows]

    if include_parent:
        all_nodes = []
        for r in results:
            current = r
            while current['parent_id'] is not None:
                cursor.execute("SELECT * FROM laws WHERE id = ?", (current['parent_id'],))
                parent = cursor.fetchone()
                if parent is None:
                    break
                parent_dict = dict(zip(columns, parent))
                all_nodes.append(parent_dict)
                current = parent_dict
        results.extend(all_nodes)

    return results[::-1]

In [6]:
with open(batch_result_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

inserted = 0

for line in tqdm(lines, desc="Inserting JSON results", unit="entry"):
    data = json.loads(line)
    if data.get("response", {}).get("status_code") != 200:
        continue

    content = data["response"]["body"]["choices"][0]["message"]["content"]
    content = content.strip()
    split_content = content.replace("\n", "").split(".")
    law_id = data["custom_id"]

    # Fetch so_hieu from original DB if needed
    law_sentence = extract_from_sqlite(gtvt_cursor, law_id)
    so_hieu = law_sentence[0]["so_hieu"] if law_sentence else "unknown"
    sequence = 1
    for c in split_content:
        c = c.strip()
        if not c:
            continue
        cursor.execute("""
            INSERT OR REPLACE INTO laws (id, sequence, content, so_hieu)
            VALUES (?, ?, ?, ?)
        """, (law_id, sequence, c, so_hieu))
        inserted += 1
        sequence += 1

conn.commit()
print(f"Inserted {inserted} records into 'laws' table")

Inserting JSON results: 100%|██████████| 943/943 [00:00<00:00, 22056.20entry/s]

Inserted 4424 records into 'laws' table


In [7]:
law_sentence = extract_from_sqlite(gtvt_cursor, "3f2f65567a03467937fa6b9f291607028d46f8809afdcbf2303e905cb8ed8ab8")
print(law_sentence)

[{'id': '3f2f65567a03467937fa6b9f291607028d46f8809afdcbf2303e905cb8ed8ab8', 'title': 'Điểm e', 'content': 'Tổng hợp chi phí vận hành công trình trạm trong 1 ca (hoặc 1 ngày) làm việc BẢNG 2.2.A - TỔNG HỢP CHI PHÍ NHÂN CÔNG, MÁY THI CÔNG, VẬT TƯ VẬT LIỆU TRONG CHI PHÍ TRỰC TIẾP VẬN HÀNH CÔNG TRÌNH TRẠM TRONG 1 ĐƠN VỊ THỜI GIAN (CA, NGÀY) STT Mã hiệu Nội dung Đơn vị Khối lượng Giá Thành tiền [1] [2] [3] [4] [5] [6] [7] = [5]x[6] I Nhân công', 'parent_id': '63cbb6d6c4f372508b3868b7ac7e5cf310b41a3c4fa8d4ca4c86b32351ca73a0', 'so_hieu': '41/2021/TT-BGTVT'}]
